# Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from google import genai
from PIL import Image
import gc
from tqdm import tqdm
import shutil
from pdf2image import convert_from_path, pdfinfo_from_path

# General settings

In [ ]:
# Input Output settings
output_pipline = "Data Extraction"
pdf_folder = r""
cropped_images_folder_name = r""
csv_output_folder_name = r""
csv_output_file_name = r""
error_files_folder = r""

# Path for resource rendering
poppler_path = r"Propper for image rendering\poppler-26.02.0\Library\bin"

# Tuning parameters settings
dpi = 500
batch_size = 5
least_horizontal_and_vertical_length = 100
thickness_thresold = 30
tolerance_of_detecting_the_horizontal_and_vertical_line_thickness = 30

# Prompt settings

In [ ]:
prompt = """# SYSTEM PROMPT: DYNAMIC UNIVERSAL SPECIFICATION EXTRACTOR, RELATIONAL MERGER & DIRECT CSV GENERATOR

You are an advanced Vision-AI and Tabular Data Normalization Specialist. Your objective is to extract technical specifications from diverse document/image tables, dynamically normalize all discovered attributes (including open-world, newly introduced features), relationally join multi-table matrices across the input batch, and produce a structurally valid RFC 4180 CSV dataset while preserving source fidelity and avoiding unsupported inference.

==================================================
1. TABLE STRUCTURE DISCOVERY & EXTRACTION
==================================================
Classify every table in the provided input into one of two structural archetypes:

TYPE A: Entity-Centric (Vertical / Single-Entity / Multi-Model Table)
- The header or top region contains the primary Entity/Model identifier(s).
- The left column contains attribute labels, and the right column contains corresponding values.
- ENTITY PRESERVATION: Treat each explicitly listed entity/model identifier as a distinct entity unless the source document explicitly groups them. Never merge, deduplicate, or abbreviate model identifiers. If a header lists multiple model identifiers separated by "/", preserve the complete header string as the entity identifier unless the table clearly assigns different specifications to individual models.
- JOIN-KEY EXTRACTION: Determine the relational join key ONLY from explicit evidence in the provided tables/documents. A value (e.g. screen inch size) may be extracted from a model identifier only when the document itself establishes that the corresponding portion represents the relevant specification. Never infer the meaning of a number or substring from general product knowledge.
- Extract all rows into canonical attribute key-value pairs.

TYPE B: Matrix-Centric (Horizontal / Cross-Reference / Multi-Column Table)
- The top-left corner cell defines the primary linking attribute.
- The first row contains discrete values for that linking attribute across multiple columns.
- Subsequent rows define complementary technical attributes per column.
- Extract each column as an independent set of complementary attributes bound strictly to its specific column identifier value.

==================================================
2. DYNAMIC & MALLEABLE SCHEMA NORMALIZATION
==================================================
1. DYNAMIC ATTRIBUTE KEY TRANSFORMATION:
   - Convert all specification labels into clean, lowercase snake_case.
   - When an attribute label includes a measurement unit in parentheses or brackets, strip the punctuation and append the unit as a descriptive suffix (e.g., parenthetical units like Hz, V, W, Kg become _hz, _v, _w, _kg).

2. SEMANTIC NORMALIZATION SAFETY BOUNDS:
   - Merge two attribute labels into the same canonical column ONLY when they unambiguously represent the same physical/technical attribute.
   - Do NOT merge attributes merely because they are related, belong to the same category, or contain similar words:
     * "net_weight_with_stand" != "net_weight_without_stand"
     * "wall_mount_dimension" != "wall_mount_screw"
     * "power_consumption_tv_on_w" != "power_consumption_standby_w"
   - When equivalence is uncertain, preserve the attributes as separate columns rather than guessing.

3. OPEN-WORLD SCHEMA PRESERVATION:
   - Never discard, omit, or collapse any newly discovered feature (e.g., refresh rates, color gamuts, operating systems, connectivity versions, HDR standards), even if it appears in only a single table or entity.

4. OCR UNCERTAINTY RULE:
   - Do not reconstruct, correct, autocomplete, or normalize visually uncertain text using domain knowledge.
   - If a character or value cannot be confidently determined from the image, preserve the visibly readable portion only if doing so does not alter its meaning; otherwise output null.
   - Never replace an uncertain value with a likely/common specification.

5. SOURCE-OF-TRUTH RULE:
   - All cell values must originate strictly from visible source content.
   - The model may generate ONLY:
     1. normalized column names;
     2. relational associations explicitly supported by matching join keys;
     3. null for missing/unavailable values.
   - The model MUST NOT generate:
     * calculated or computed values;
     * corrected values;
     * inferred specifications;
     * assumed units;
     * values copied from similar models;
     * values derived from general product knowledge.

6. ZERO-HALLUCINATION & STRICT NULL POLICY:
   - Extract ONLY data that is visibly and explicitly present.
   - Never infer, predict, interpolate, or extrapolate missing specifications. If an entity lacks a value or has no corresponding entry in a complementary matrix, record its value strictly as null.

7. VERBATIM TEXTUAL FIDELITY:
   - Preserve raw textual formatting, mathematical expressions, compound dimensions, ranges, and comparison operators exactly as printed. Do not evaluate, reduce, or compute mathematical expressions.

==================================================
3. RELATIONAL MERGING & CSV GENERATION (RFC 4180)
==================================================
1. RELATIONAL JOIN:
   - Relationally join Type B complementary columns to Type A model rows strictly where their explicit join keys match identically.
   - ANTI-COLLISION DIRECTIVE: Never join or match columns against unrelated numerical attributes (e.g., electrical power ratings, frequencies, or weight values).

2. CSV COLUMN HEADER (UNION OF ALL ATTRIBUTES):
   - Row 1 must be the complete mathematical union of all unique normalized attributes discovered across all processed tables.
   - Order columns logically:
     1. Primary Entity Identifier (`model`)
     2. Primary Relational Key (`tv_inch_size` or corresponding linking key)
     3. Core Functional & Display Specifications
     4. Electrical & Power Specifications
     5. Audio, System & Connectivity Specifications
     6. Physical Dimensions & Weights
     7. Complementary Mounting & Structural Specifications
     8. Packaging & Logistics Specifications
     9. Any additional dynamic / novel specifications

3. CSV ROW FORMATTING:
   - Exactly one row per distinct entity.
   - Every column must strictly align with its respective header.
   - Missing or unlinked attributes must be written as `null`.
   - Any value containing commas, double quotes, or newlines must be enclosed in standard double quotes `""`. Double quotes inside values must be escaped as `""`.

==================================================
4. STRICT OUTPUT DIRECTIVE
==================================================
Return ONLY the raw CSV text.
Do NOT include markdown formatting, code block backticks (```csv), commentary, explanations, or introductory text. Start directly with the CSV header row."""

# Functions for pipline

In [ ]:
# convert_the_image_to_inverted black and white image
def convert_to_inverted_black_and_white(image):
  gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  _, bw_image = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
  if (bw_image == 0).sum() > (bw_image.size / 2):
    sol = bw_image
  else:
    sol = cv2.bitwise_not(bw_image)
  sol = Image.fromarray(sol)
  return sol

In [ ]:
def get_horizontal_and_vertical_lines (img, least_horizontal_and_vertical_length = 50):
  # if img.size == 0:
  #   return -1
  horizontal_kernel_size = max(least_horizontal_and_vertical_length, img.shape[1] // least_horizontal_and_vertical_length)
  horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (horizontal_kernel_size, 1))
  vertical_kernel_size = max(least_horizontal_and_vertical_length, img.shape[0] // least_horizontal_and_vertical_length)
  vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,vertical_kernel_size))
  
  horizontal_lines = cv2.morphologyEx(img, cv2.MORPH_OPEN, horizontal_kernel)
  vertical_lines = cv2.morphologyEx(img, cv2.MORPH_OPEN, vertical_kernel)
  
  # plt.imshow(cv2.bitwise_or(horizontal_lines, vertical_lines), cmap='gray')
  # plt.show()
  
  return horizontal_lines, vertical_lines

In [ ]:
def check_the_cropped_image_validity_to_table(img, tolerance, thickness_thresold):
  lines = get_horizontal_and_vertical_lines(img, least_horizontal_and_vertical_length=least_horizontal_and_vertical_length)
  if lines == -1:
    print ("This line is ")
    return -1
  horizontal_lines, vertical_lines = lines
  horizontal_projection = np.sum(horizontal_lines > 0, axis=1)
  horizontal_indexes = np.where(horizontal_projection > 0)[0]
  vertical_projection = np.sum(vertical_lines, axis = 0)
  vertical_indexes = np.where(vertical_projection > 0)[0]
  
  horizontal_diff = np.diff(horizontal_indexes)
  vertical_diff = np.diff(vertical_indexes)
  
  split_horizontal_index = np.where(horizontal_diff > tolerance)[0] + 1
  split_vertical_index = np.where(vertical_diff > tolerance)[0] + 1
  
  split_horizontal_index = np.split(horizontal_indexes, split_horizontal_index)
  split_vertical_index = np.split(vertical_indexes, split_vertical_index)
  horizontal_thicknesses = np.array(list(map(lambda x: len(x), split_horizontal_index)))
  vertical_thickness = np.array(list(map(lambda x: len(x), split_vertical_index)))
  if np.any(horizontal_thicknesses > thickness_thresold):
    return -1
  if np.any(vertical_thickness > thickness_thresold):
    return -1
  return None

In [ ]:
def the_number_of_rows_and_columns(horizontal_lines, vertical_lines):
  # My problem solving optimization
  row_projection = np.sum(horizontal_lines > 0, axis=1)
  row_positions = np.where(row_projection > 0)[0]
  number_of_rows = len(set(row_positions+range(len(row_positions),0,-1))) - 1

  column_projection = np.sum(vertical_lines > 0, axis=0)
  column_positions = np.where(column_projection > 0)[0]
  number_of_columns = len(set(column_positions+range(len(column_positions),0,-1))) - 1
  return number_of_rows, number_of_columns

In [ ]:
def table_state(number_of_horizontals, number_of_verticals, contour_nums):
  if number_of_horizontals < 2 or number_of_verticals < 2:
    return -1
  if (number_of_horizontals+1) * (number_of_verticals+1) == contour_nums:
    return True
  if ((number_of_horizontals+1) * (number_of_verticals+1)) - 1 == contour_nums:
    return False
  return -1

In [ ]:
def detect_and_crop_tables(image, output_folder, table_count = 0):
    if image is None:
        raise ValueError("image is None")

    if len(image.shape) != 2:
        raise ValueError("image must be a grayscale/binary image")

    os.makedirs(output_folder, exist_ok=True)
    binary = image
    horizontal_lines, vertical_lines = get_horizontal_and_vertical_lines(binary, least_horizontal_and_vertical_length=least_horizontal_and_vertical_length)
    # print (horizontal_lines)
    table_structure = np.zeros_like(binary)

    cv2.bitwise_or(table_structure, horizontal_lines, dst=table_structure)
    del horizontal_lines
    gc.collect()

    cv2.bitwise_or(table_structure, vertical_lines, dst=table_structure)

    del vertical_lines
    gc.collect()

    connect_kernel = cv2.getStructuringElement(cv2.MORPH_RECT,(5, 5))

    cv2.dilate(table_structure, connect_kernel, dst=table_structure, iterations=2)
    del connect_kernel
    gc.collect()

    contours, _ = cv2.findContours(table_structure, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    del table_structure
    gc.collect()
    boxes = []

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h
        # Ignore very small objects
        if area < 5000:
            continue
        if w < 100 or h < 50:
            continue
        boxes.append((y, x, w, h))
    boxes.sort()
    padding = 10
    for y, x, w, h in boxes:
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(binary.shape[1], x + w + padding)
        y2 = min(binary.shape[0], y + h + padding)
        # This is a VIEW, not a full image copy
        table_crop = binary[y1:y2, x1:x2]
        # plt.imshow(table_crop, cmap='gray')
        # plt.show()
        lines = get_horizontal_and_vertical_lines(table_crop, least_horizontal_and_vertical_length=least_horizontal_and_vertical_length)
        if lines == -1:
            continue
        horizontal_lines, vertical_lines = lines
        if check_the_cropped_image_validity_to_table(table_crop, tolerance_of_detecting_the_horizontal_and_vertical_line_thickness, thickness_thresold) == -1:
            # print ("The number of rows and columns")
            # plt.imshow(cv2.bitwise_or(horizontal_lines, vertical_lines), cmap='gray')
            # plt.show()
            # print ("This is not a table")
            continue
        #Problem solving optimization
        number_of_rows, number_of_columns = the_number_of_rows_and_columns(horizontal_lines, vertical_lines)
        

        intersection = cv2.bitwise_and(vertical_lines, horizontal_lines)
        contours, _ = cv2.findContours(intersection, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        number_of_intersection_points = len(contours)
        if table_state(number_of_rows, number_of_columns, number_of_intersection_points) == -1:
            continue
        table_count += 1
        output_path = os.path.join(output_folder, f"table_{table_count}.png")
        cv2.imwrite(output_path, table_crop)
        print( f"Table {table_count}: " f"x={x1}, y={y1}, " f"width={x2 - x1}, " f"height={y2 - y1}")
    del contours
    del boxes
    gc.collect()

    print("--------------------------------")
    print("Tables detected:", table_count)
    print("Output folder:", output_folder)
    print("--------------------------------")

    return table_count
image = cv2.imread(r"D:\Programming\Projects\Learning python librairies\OCR\Outline_Tables\sample\sample\sample_page_5.png", cv2.IMREAD_GRAYSCALE)
detect_and_crop_tables(image, r"D:\Programming\Projects\Learning python librairies\OCR\cropped_tables")

In [ ]:
def csv_file_product_images(images):
  client = genai.Client(api_key = "")
  response = client.models.generate_content(model = "gemini-3.1-flash-lite", contents = [prompt, images])
  return response

# Pipline for the product

In [ ]:
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]
errors_count = 0
for file in pdf_files:
  pdf_path = os.path.join(pdf_folder, file)
  pdf_file_name = os.path.splitext(file)[0]
  pdf_folder_output = os.path.join(output_pipline, pdf_file_name)
  os.makedirs(pdf_folder_output, exist_ok=True)
  pdf_pages_folder = os.path.join(pdf_folder_output, "pdf_pages")
  os.makedirs(pdf_pages_folder, exist_ok=True)
  pdf_csv_output_folder = os.path.join(pdf_folder_output, csv_output_folder_name)
  os.makedirs(pdf_csv_output_folder, exist_ok=True)
  pages_cropped_table_folder = os.path.join(pdf_folder_output, cropped_images_folder_name)
  os.makedirs(pages_cropped_table_folder, exist_ok=True)
  
  # Get the total number of pages in the PDF
  info = pdfinfo_from_path(pdf_path,poppler_path=poppler_path)
  total_pages = int(info["Pages"])
  
  for batch_number in range(1, (total_pages // batch_size)+1):
    data = convert_from_path(pdf_path, first_page=batch_number*batch_size-batch_size+1, last_page=batch_number*batch_size, dpi=dpi, poppler_path=poppler_path, thread_count=os.cpu_count())
    for i, img in enumerate(data):
      page = convert_to_inverted_black_and_white(cv2.cvtColor(np.array(page), cv2.COLOR_RGB2BGR))
      page.save(os.path.join(pdf_pages_folder, f"{pdf_file_name}_page_{batch_number*batch_size-batch_size+i+1}.png"), "png")
      table_count = detect_and_crop_tables(np.array(page), pages_cropped_table_folder, table_count=table_count)
    del data
    gc.collect()
  
  remaining_pages = total_pages % batch_size
  if remaining_pages > 0:
    data = convert_from_path(pdf_path, first_page=total_pages-remaining_pages+1, last_page=total_pages, dpi=dpi, poppler_path=poppler_path, thread_count=os.cpu_count())
    for i, page in enumerate(data):
      page = convert_to_inverted_black_and_white(cv2.cvtColor(np.array(page), cv2.COLOR_RGB2BGR))
      page.save(os.path.join(pdf_pages_folder, f"{pdf_file_name}_page_{total_pages-remaining_pages+1+i}.png"), "png")
      table_count = detect_and_crop_tables(np.array(page), pages_cropped_table_folder, table_count=table_count)
    del data
    gc.collect()
  
  # Path the cropped files to the model to get the output
  images_for_model = []
  for file in os.listdir(pages_cropped_table_folder):
    if not file.endswith('.png'):
      continue
    img = Image.open(os.path.join(pages_cropped_table_folder, file))
    images_for_model.append(img)
  csv_response = csv_file_product_images(images_for_model)
  with open(os.path.join(pdf_csv_output_folder, "output.csv"), 'w', encoding='utf-8') as f:
    f.write(csv_response.output_text)